# Student Performance Analysis & Modeling
This notebook inspects, cleans, visualizes, and models student performance data. It covers exploratory data analysis (EDA), correlation, vector operations, and feature scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

n = 100
study_hours = np.random.normal(5, 2, n)
attendance = np.random.normal(85, 10, n)
previous_mark = np.random.normal(70, 15, n)
final_score = (0.5 * study_hours) + (0.3 * attendance) + (0.2 * previous_mark) + np.random.normal(0, 5, n)

df = pd.DataFrame({
    'study_hours': study_hours,
    'attendance': attendance,
    'previous_mark': previous_mark,
    'final_score': final_score
})

# Introduce data issues as specified
df.loc[0:5, 'study_hours'] = np.nan # Missing values
df.loc[99, 'study_hours'] = 50 # Outlier
df.loc[98, 'attendance'] = -10 # Impossible value

print("Original Data Sample:")
print(df.head(10))

## Part A — Inspect and Clean
1. **Outlier & Data Issues Identification**: 
   - `study_hours` contains missing values (`NaN`) in rows 0-5 and an extreme outlier of `50` hours at row 99 (given a normal distribution mean of 5).
   - `attendance` contains an impossible negative value (`-10`) at row 98.
   - **Handling Strategy**: We will impute missing `study_hours` with the median, cap the outlier at a reasonable threshold, and replace negative attendance with the median attendance.
2. **Feature Matrix Shape & Types**:
   - Numerical columns: `study_hours`, `attendance`, `previous_mark`.
   - Target variable: `final_score`.

In [ ]:
# Handle missing values in study_hours using median
study_hours_median = df['study_hours'].median()
df['study_hours'].fillna(study_hours_median, inplace=True)

# Handle impossible attendance value (-10) using median
attendance_median = df[df['attendance'] >= 0]['attendance'].median()
df.loc[df['attendance'] < 0, 'attendance'] = attendance_median

# Handle outlier in study_hours (50 hours -> cap at 95th percentile)
upper_limit = df['study_hours'].quantile(0.95)
df['study_hours'] = np.where(df['study_hours'] > 15, upper_limit, df['study_hours'])

# Feature Matrix Shape
X = df[['study_hours', 'attendance', 'previous_mark']]
y = df['final_score']

print(f"Feature Matrix Shape (X): {X.shape}")
print(f"Target Variable: final_score")

## Part B — Summarize
1. **Statistical Summary**: Mean, median, and standard deviation calculated across all cleaned columns.
2. **Observations**:
   - **Consistent Attendance**: The standard deviation for attendance is relatively low compared to its mean, showing that most students maintain steady attendance rates.
   - **Academic Diversity**: Previous marks show a wide spread (high standard deviation), highlighting diverse starting academic backgrounds.
   - **Study Habits**: Most students cluster around 4 to 6 study hours per week.

In [ ]:
summary_stats = pd.DataFrame({
    'Mean': df.mean(),
    'Median': df.median(),
    'Standard Deviation': df.std()
})

print(summary_stats)

## Part C — Visualize
1. **Histogram of Final Scores**: Displays the distribution and frequency of final scores.
2. **Scatter Plot (Study Hours vs. Final Score)**: Shows a positive linear trend—students who dedicate more study hours generally achieve higher final scores.

In [ ]:
plt.figure(figsize=(12, 5))

# Histogram for final_score
plt.subplot(1, 2, 1)
sns.histplot(df['final_score'], kde=True, color='skyblue')
plt.title('Distribution of Final Scores')
plt.xlabel('Final Score')
plt.ylabel('Frequency')

# Scatter plot: study_hours vs final_score
plt.subplot(1, 2, 2)
sns.scatterplot(x='study_hours', y='final_score', data=df, color='purple')
plt.title('Study Hours vs Final Score')
plt.xlabel('Study Hours')
plt.ylabel('Final Score')

plt.tight_layout()
plt.show()

## Part D — Correlation and Causality
1. **Pearson Correlation**: Measures the linear association between study hours and final scores.
2. **Critical Thinking**: 
   - **Does correlation guarantee a specific score?** No. Correlation measures strength and direction of a linear relationship, not a deterministic guarantee. Individual variance (test anxiety, sleep, luck) still plays a role.
   - **Lurking/Confounding Variable**: *Student motivation level* or *prior subject aptitude*. A highly motivated student might study more *and* absorb material faster, driving both high study hours and high scores.

In [ ]:
correlation = df['study_hours'].corr(df['final_score'])
print(f"Pearson Correlation between study_hours and final_score: {correlation:.4f}")

## Part E — Vectors and Dot Product
1. **Vector x**: Selected from one student row (excluding target).
2. **Weights & Bias**: Weight vector $w = [0.5, 0.2, 0.4]$, bias $b = 5$.
3. **Concept**: 
   - $w$ represents the learned feature weights (coefficients) indicating how much each feature contributes to the final predicted score.
   - $b$ (bias/intercept) represents the baseline score expected even if all input features are zero.

In [ ]:
# Select one student row features as a vector x
x = X.iloc[10].values # e.g., student at index 10

# Define weights and bias
w = np.array([0.5, 0.2, 0.4])
b = 5

# Calculate y using dot product
y_pred = np.dot(w, x) + b

print(f"Selected Student Vector (x): {x}")
print(f"Calculated Score (y): {y_pred:.2f}")

## Part F — Scaling
1. **Standardization Formula**: $z = \frac{x - \mu}{\sigma}$ applied to `study_hours` and `previous_mark`.
2. **Why Distance-Based Algorithms Fail Without Scaling**:
   - Features like `previous_mark` range from 20 to 100+, whereas `study_hours` range from 0 to 10.
   - In algorithms like **K-Nearest Neighbors (KNN)** which rely on Euclidean distance, features with larger numerical ranges would completely dominate the distance calculation, making smaller-scale features effectively invisible.

In [ ]:
# Standardizing study_hours and previous_mark
df['study_hours_scaled'] = (df['study_hours'] - df['study_hours'].mean()) / df['study_hours'].std()
df['previous_mark_scaled'] = (df['previous_mark'] - df['previous_mark'].mean()) / df['previous_mark'].std()

print(df[['study_hours', 'study_hours_scaled', 'previous_mark', 'previous_mark_scaled']].head())